In [2]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = "Papa"
STUDENT_ID = "TUPM-26-1927"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

TARGET_YEAR = 2015 + SEED_DIGIT
LOSS_FIELD = "Financial Loss (in Million $)"
COUNT_FIELD = "Attack Type"

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Target year:", TARGET_YEAR)
print("Workspace:", Path(".").resolve())

Seed: 7
ID Sum: 27
Target year: 2022
Workspace: C:\Users\Cyrus\OneDrive\Desktop\Lab6_File_Handling\Lab6_File_Handling_Papa\Lab6_File_Handling_Additional_Exercises


In [3]:
# [CELL 2] INSPECT THE DATASET
import csv

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
print("Exists:", dataset_path.exists())

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.reader(file)
    header = next(reader)
    print("Columns:")
    for index, name in enumerate(header):
        print(f"  {index}: {name!r}")
    print("First row:", next(reader))

Exists: True
Columns:
  0: 'Country'
  1: 'Year'
  2: 'Attack Type'
  3: 'Target Industry'
  4: 'Financial Loss (in Million $)'
  5: 'Number of Affected Users'
  6: 'Attack Source'
  7: 'Security Vulnerability Type'
  8: 'Defense Mechanism Used'
  9: 'Incident Resolution Time (in Hours)'
First row: ['China', '2019', 'Phishing', 'Education', '80.53', '773169', 'Hacker Group', 'Unpatched Software', 'VPN', '63']


In [4]:
# [CELL 3] ANALYZE, EXPORT, REPORT, AND BACK UP
import json
import shutil

YEAR_FIELD = "Year"
REQUIRED_FIELDS = [YEAR_FIELD, COUNT_FIELD, LOSS_FIELD]

total_records = 0
invalid_records = 0
selected = []          # incidents in the target year, in original dataset order
attack_counts = {}
total_loss = 0.0

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    missing = [name for name in REQUIRED_FIELDS if name not in reader.fieldnames]
    if missing:
        raise KeyError(f"Missing columns: {missing}. Found: {reader.fieldnames}")

    fieldnames = reader.fieldnames

    for row in reader:
        total_records += 1

        try:
            year = int(row[YEAR_FIELD])
        except (ValueError, TypeError):
            invalid_records += 1
            continue

        if year != TARGET_YEAR:
            continue

        try:
            loss = float(row[LOSS_FIELD])
        except (ValueError, TypeError):
            invalid_records += 1
            continue

        selected.append(row)
        attack = row[COUNT_FIELD].strip()
        attack_counts[attack] = attack_counts.get(attack, 0) + 1
        total_loss += loss

selected_count = len(selected)
average_loss = total_loss / selected_count if selected_count else 0.0

# Five costliest: highest loss first; sorted() is stable, so ties keep dataset order
top_five = sorted(selected, key=lambda r: float(r[LOSS_FIELD]), reverse=True)[:5]

# 1. Filtered CSV of all incidents in the target year
filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_filtered.csv"
with open(filtered_csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected)

# 2. JSON file with the five costliest incidents and the summary
summary_json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_summary.json"
summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "target_year": TARGET_YEAR,
    "total_records": total_records,
    "invalid_records": invalid_records,
    "incidents_in_target_year": selected_count,
    "count_per_attack_type": dict(sorted(attack_counts.items())),
    "total_financial_loss_million_usd": round(total_loss, 2),
    "average_financial_loss_million_usd": round(average_loss, 2),
    "five_costliest_incidents": top_five,
    "note": "" if selected_count else "No incidents found in the target year.",
}
with open(summary_json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# 3. Back up the JSON file and verify the copy
backup_path = BACKUP_DIR / summary_json_path.name
shutil.copy2(summary_json_path, backup_path)

with open(backup_path, "r", encoding="utf-8") as file:
    backup_matches = json.load(file) == summary
backup_verified = (
    backup_path.exists()
    and backup_path.stat().st_size == summary_json_path.stat().st_size
    and backup_matches
)

# 4. Text report
report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_report.txt"
with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-----------------------------\n")
    report.write(f"Last Name: {LAST_NAME}\n")
    report.write(f"Student ID: {STUDENT_ID}\n")
    report.write(f"Target Year: {TARGET_YEAR}\n")
    report.write(f"Total Records in Dataset: {total_records}\n")
    report.write(f"Invalid Records Skipped: {invalid_records}\n")
    report.write(f"Incidents in Target Year: {selected_count}\n\n")
    report.write("Incidents per Attack Type:\n")
    for attack, count in sorted(attack_counts.items()):
        report.write(f"  {attack}: {count}\n")
    report.write(f"\nTotal Financial Loss (USD millions): {total_loss:.2f}\n")
    report.write(f"Average Financial Loss (USD millions): {average_loss:.2f}\n\n")
    report.write("Five Costliest Incidents:\n")
    if top_five:
        for rank, row in enumerate(top_five, 1):
            report.write(
                f"  {rank}. {row[COUNT_FIELD]} | {float(row[LOSS_FIELD]):.2f}M"
                f" | {row.get('Country', 'N/A')} | {row.get('Target Industry', 'N/A')}\n"
            )
    else:
        report.write("  (no incidents in the target year)\n")
    report.write("\nOutput Files:\n")
    report.write(f"  Filtered CSV: {filtered_csv_path.name}\n")
    report.write(f"  JSON Summary: {summary_json_path.name}\n")
    report.write(f"  Backup: {backup_path.name} (verified: {backup_verified})\n")

# 5. Display the results
print("Target year:", TARGET_YEAR)
print("Total records in dataset:", total_records)
print("Invalid records skipped:", invalid_records)
print("Incidents in target year:", selected_count)
print("Count per attack type:", dict(sorted(attack_counts.items())))
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(average_loss, 2))
print("Five costliest incidents:")
for rank, row in enumerate(top_five, 1):
    print(f"  {rank}. {row[COUNT_FIELD]} | {float(row[LOSS_FIELD]):.2f}M | {row.get('Country', 'N/A')}")
print("Filtered CSV:", filtered_csv_path)
print("JSON summary:", summary_json_path)
print("Backup verification:", backup_verified)
print("Report file:", report_path)

Target year: 2022
Total records in dataset: 3000
Invalid records skipped: 0
Incidents in target year: 318
Count per attack type: {'DDoS': 54, 'Malware': 64, 'Man-in-the-Middle': 41, 'Phishing': 57, 'Ransomware': 56, 'SQL Injection': 46}
Total financial loss (USD millions): 15870.86
Average financial loss (USD millions): 49.91
Five costliest incidents:
  1. Phishing | 98.47M | Japan
  2. SQL Injection | 98.46M | USA
  3. Phishing | 98.06M | Brazil
  4. Phishing | 97.56M | UK
  5. Malware | 97.32M | Japan
Filtered CSV: output\Papa_TUPM-26-1927_cyber_filtered.csv
JSON summary: output\Papa_TUPM-26-1927_cyber_summary.json
Backup verification: True
Report file: output\Papa_TUPM-26-1927_cyber_report.txt
